In [1]:
import mysql.connector
import pandas as pd
import numpy as np

conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='12345',
    database='nykaa_campaign_db'
)

query = "SELECT * FROM nykaa_campaign_data"
df = pd.read_sql(query, conn)
conn.close()

print(df.head())
print(df.shape)

C:\Users\Aarya Khire\AppData\Local\Temp\ipykernel_12512\605148607.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


   Campaign_ID Campaign_Type        Target_Audience  Duration  \
0  NY-CMP-1000  Social Media       College Students        21   
1  NY-CMP-1001      Paid Ads  Tier 2 City Customers        18   
2  NY-CMP-1002    Influencer                  Youth        23   
3  NY-CMP-1003         Email          Working Women        18   
4  NY-CMP-1004      Paid Ads       College Students        10   

                   Channel_Used  Impressions  Clicks  Leads  Conversions  \
0             WhatsApp, YouTube        57804    6156   3616         2355   
1                       YouTube        91801    3321   1971         1357   
2     WhatsApp, Google, YouTube        15536    2182    952          755   
3  YouTube, Facebook, Instagram        88114    8413   2231          947   
4           Facebook, Instagram        96871    3743   2060         1258   

   Revenue  Acquisition_Cost   ROI Language  Engagement_Score  \
0  1867515            111.03  6.14    Hindi             20.98   
1  1046247            

In [4]:
# Data Cleaning
df.columns = df.columns.str.strip()

text_cols = [
    'Campaign_ID', 'Campaign_Type', 'Target_Audience',
    'Channel_Used', 'Language', 'Customer_Segment'
]

for col in text_cols:
    df[col] = df[col].astype('string').str.strip()

num_cols = [
    'Duration', 'Impressions', 'Clicks', 'Leads',
    'Conversions', 'Revenue', 'Acquisition_Cost',
    'ROI', 'Engagement_Score'
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['date_new'] = pd.to_datetime(df['date_new'], errors='coerce')

In [5]:
# Create KPIs
df['CTR_%'] = np.where(df['Impressions'] > 0, df['Clicks'] * 100 / df['Impressions'], np.nan)
df['Conversion_Rate_%'] = np.where(df['Clicks'] > 0, df['Conversions'] * 100 / df['Clicks'], np.nan)
df['Lead_to_Conversion_%'] = np.where(df['Leads'] > 0, df['Conversions'] * 100 / df['Leads'], np.nan)
df['Revenue_per_Cost'] = np.where(df['Acquisition_Cost'] > 0, df['Revenue'] / df['Acquisition_Cost'], np.nan)
df['Month'] = df['date_new'].dt.to_period('M').astype(str)
df['Week'] = df['date_new'].dt.to_period('W').astype(str)

In [6]:
campaign_summary = (
    df.groupby('Campaign_Type', as_index=False)
      .agg(
          total_revenue=('Revenue', 'sum'),
          avg_roi=('ROI', 'mean'),
          avg_ctr=('CTR_%', 'mean'),
          avg_conversion_rate=('Conversion_Rate_%', 'mean'),
          avg_engagement=('Engagement_Score', 'mean')
      )
      .sort_values('total_revenue', ascending=False)
)

print(campaign_summary.head(10))

  Campaign_Type  total_revenue   avg_roi   avg_ctr  avg_conversion_rate  \
1    Influencer     5769064044  2.699687  8.560704            22.005118   
4  Social Media     5751837620  2.754784  8.539556            21.920567   
2      Paid Ads     5751468983  2.722275  8.513291            21.806196   
3           SEO     5698831847  2.709630  8.454946            22.046932   
0         Email     5685161788  2.682663  8.474684            22.079182   

   avg_engagement  
1       13.861539  
4       13.845957  
2       13.751833  
3       13.713328  
0       13.747823  


In [7]:
# Channel performance
channel_summary = (
    df.groupby('Channel_Used', as_index=False)
      .agg(
          total_revenue=('Revenue', 'sum'),
          total_conversions=('Conversions', 'sum'),
          avg_roi=('ROI', 'mean'),
          avg_ctr=('CTR_%', 'mean'),
          avg_conversion_rate=('Conversion_Rate_%', 'mean')
      )
      .sort_values('avg_roi', ascending=False)
)

print(channel_summary)

                    Channel_Used  total_revenue  total_conversions   avg_roi  \
17     Email, WhatsApp, Facebook       92965505             174572  4.230621   
142       YouTube, Google, Email       91618963             174046  3.960405   
62     Google, Facebook, YouTube       80265762             162731  3.600588   
39   Facebook, Instagram, Google       82550963             155437  3.546835   
97   Instagram, WhatsApp, Google       91465465             188876  3.507256   
..                           ...            ...                ...       ...   
140  YouTube, Facebook, WhatsApp       68183035             138908  1.986525   
112   WhatsApp, Facebook, Google       82614265             159180  1.981844   
121   WhatsApp, Instagram, Email       67687593             136618  1.928228   
65   Google, Instagram, Facebook       58059912             125124  1.873357   
109     WhatsApp, Email, YouTube       65282827             124083  1.811733   

      avg_ctr  avg_conversion_rate  
17

In [8]:
 #Audience and segment performance
segment_summary = (
    df.groupby(['Target_Audience', 'Customer_Segment'], as_index=False)
      .agg(
          total_revenue=('Revenue', 'sum'),
          total_conversions=('Conversions', 'sum'),
          avg_roi=('ROI', 'mean'),
          avg_engagement=('Engagement_Score', 'mean')
      )
      .sort_values('total_revenue', ascending=False)
)

print(segment_summary.head(15))

          Target_Audience       Customer_Segment  total_revenue  \
0        College Students       College Students     1247851346   
8        Premium Shoppers          Working Women     1212302981   
16          Working Women       Premium Shoppers     1195478826   
9        Premium Shoppers                  Youth     1176911410   
5        Premium Shoppers       College Students     1172128031   
13  Tier 2 City Customers          Working Women     1167646953   
2        College Students  Tier 2 City Customers     1160668528   
19          Working Women                  Youth     1159399623   
3        College Students          Working Women     1158275528   
18          Working Women          Working Women     1156948184   
7        Premium Shoppers  Tier 2 City Customers     1148038815   
12  Tier 2 City Customers  Tier 2 City Customers     1146886052   
6        Premium Shoppers       Premium Shoppers     1142903708   
14  Tier 2 City Customers                  Youth     114162896

In [9]:
# Monthly trend analysis
monthly_trend = (
    df.groupby('Month', as_index=False)
      .agg(
          total_revenue=('Revenue', 'sum'),
          total_conversions=('Conversions', 'sum'),
          avg_roi=('ROI', 'mean'),
          avg_ctr=('CTR_%', 'mean')
      )
      .sort_values('Month')
)

print(monthly_trend)

      Month  total_revenue  total_conversions   avg_roi   avg_ctr
0   2024-07     2630226313            5224622  2.809377  8.566229
1   2024-08     2597876736            5187225  2.767831  8.580008
2   2024-09     2390680030            4791725  2.672931  8.415942
3   2024-10     2583203421            5193529  2.616831  8.495032
4   2024-11     2523763004            5070569  2.748278  8.526272
5   2024-12     2586353806            5148613  2.712556  8.544393
6   2025-01     2521812438            5028996  2.654092  8.465031
7   2025-02     2333105678            4716303  2.758309  8.551767
8   2025-03     2570717938            5139085  2.733766  8.504062
9   2025-04     2439364737            4951704  2.644182  8.505611
10  2025-05     2572782370            5082964  2.777322  8.484141
11  2025-06      906477811            1845587  2.599144  8.389409


In [10]:
# Weekly trend analysis
weekly_trend = (
    df.groupby('Week', as_index=False)
      .agg(
          total_revenue=('Revenue', 'sum'),
          total_conversions=('Conversions', 'sum'),
          avg_roi=('ROI', 'mean')
      )
      .sort_values('Week')
)

print(weekly_trend.head(20))

                     Week  total_revenue  total_conversions   avg_roi
0   2024-07-01/2024-07-07      556643244            1127190  2.721514
1   2024-07-08/2024-07-14      614144979            1216592  2.736074
2   2024-07-15/2024-07-21      605614440            1186198  2.767560
3   2024-07-22/2024-07-28      603364686            1213885  3.059955
4   2024-07-29/2024-08-04      588614036            1163981  2.577783
5   2024-08-05/2024-08-11      555395594            1119781  2.876317
6   2024-08-12/2024-08-18      577749837            1130655  2.753092
7   2024-08-19/2024-08-25      615163608            1238420  2.768606
8   2024-08-26/2024-09-01      596474150            1189583  2.808852
9   2024-09-02/2024-09-08      520552384            1053822  2.615789
10  2024-09-09/2024-09-15      571203419            1123781  2.665685
11  2024-09-16/2024-09-22      568242992            1154509  2.729909
12  2024-09-23/2024-09-29      568690465            1135706  2.692455
13  2024-09-30/2024-

In [12]:
# High spend, low return campaigns

low_return = df.sort_values(
    ['Acquisition_Cost', 'ROI'],
    ascending=[False, True]
)[['Campaign_ID', 'Campaign_Type', 'Acquisition_Cost', 'ROI', 'Revenue']].head(10)

print(low_return)

        Campaign_ID Campaign_Type  Acquisition_Cost   ROI  Revenue
33324  NY-CMP-34324      Paid Ads          15473.16 -0.95    14079
15583  NY-CMP-16583      Paid Ads          12423.65 -0.96     8920
51344  NY-CMP-52344      Paid Ads          10920.48 -0.93    19175
2875    NY-CMP-3875    Influencer           9661.16 -0.93    20801
302     NY-CMP-1302    Influencer           9469.83 -0.93    16944
22760  NY-CMP-23760    Influencer           9438.00 -0.95    11674
49497  NY-CMP-50497         Email           8983.24 -0.95    12006
53089  NY-CMP-54089      Paid Ads           8703.74 -0.95    15912
19609  NY-CMP-20609           SEO           8686.46 -0.95    11760
27468  NY-CMP-28468         Email           8414.39 -0.92    18732
